# EXP-2026-008 C4 follow-up 4 on LUMIERE: the cavity level (CPU)

Would a supplied cavity class, with a typical fill level for that class, fix the scaffold's losses? The classes are "dark", or "not dark on T1" like the blood-like cores found earlier. And is a flat fill level enough at all? This notebook runs the check fixed in card `experiments/registry/EXP-2026-008.md` (C4 follow-up 4) on the same 25 accepted training-role pairs of EXP-2026-005 run r2. Every step calls the tested module `rhuh_synthesis/lumiere_scaffold_check.py` (`run_level_check`, which uses `rhuh_synthesis/cavity_level.py`).

- **What it measures:** for each pair, the C4 primary-region error of five fills of the pre-op core:
  - the copy and the dark scaffold, as in C4;
  - the pair's own real post-op level, an upper bound that uses the real post-op scan;
  - the typical level of the other pairs of the same class;
  - the typical level of all other pairs, with no class.

  The class comes from follow-up 2's record: "not dark" if the post-op core is above 0 on T1.
- **Inputs:**
  - run r2's records and training-role exports;
  - the records of runs `c4_lumiere_regions_v1` and `c4_lumiere_signals_v1`.

  All are read through the Drive API by ID, and each file is checked by MD5. Those folders are never written. No LUMIERE image file and no development export is read.
- **Checks:** the cohort is checked as in C4. Both records must match the split and the pairs, and each pair's copy and dark scores must equal the region record's. Anything else stops the run.
- **Output:** one write-once record, uploaded through the Drive API to `MyDrive/LUMIERE_derived/EXP-2026-008/<RUN_TAG>` and confirmed by MD5. The record names patients, so keep that folder private. The printed summary and table name no one.

A CPU runtime is enough. Expect about 5 to 10 minutes, the download included. Colab asks once for permission to use the Drive API. Keep the tab open until the last cell finishes.

**Clear all outputs before saving, sharing or committing this notebook.**

In [ ]:
REPO_URL = "https://github.com/AIVIETNAM-AIO-Tuan/MedSegFactory.git"  # your fork only
BRANCH = "lumiere-onboarding"
WORK = "/content/MedSegFactory-c4-level"
RUN_TAG = "c4_lumiere_level_v1"  # a new tag for every attempt; a run folder that holds files is refused
SOURCE_RUN = ("LUMIERE_derived", "EXP-2026-005", "r2")  # read only
REGIONS_RUN = ("LUMIERE_derived", "EXP-2026-008", "c4_lumiere_regions_v1")  # read only
SIGNALS_RUN = ("LUMIERE_derived", "EXP-2026-008", "c4_lumiere_signals_v1")  # read only
OUT_RUN = ("LUMIERE_derived", "EXP-2026-008", RUN_TAG)  # created under MyDrive through the Drive API
EXPECTED_PAIRS = 25
LOCAL_SOURCE = "/content/c4_source_r2"
LOCAL_REGIONS = "/content/c4_lumiere_regions_v1_record"
LOCAL_SIGNALS = "/content/c4_lumiere_signals_v1_record"
LOCAL_RUN = f"/content/{RUN_TAG}"

## 1. Code from the fork, packages, Drive API, run folders

In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

assert not Path(LOCAL_RUN).exists(), f"{LOCAL_RUN} exists: choose a new RUN_TAG or a fresh runtime"
if not Path(WORK).is_dir():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, WORK], check=True)
else:
    current = subprocess.check_output(["git", "-C", WORK, "branch", "--show-current"], text=True).strip()
    assert current == BRANCH, "This checkout belongs to another task; choose a different WORK"
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
os.chdir(WORK)
sys.path.insert(0, WORK)

from rhuh_synthesis.colab_packages import DATA  # the one package list; it imports nothing

subprocess.run([sys.executable, "-m", "pip", "install", "-q", *DATA], check=True)
commit = subprocess.check_output(["git", "log", "-1", "--oneline"], text=True).strip()
print(commit)

from google.colab import auth
from googleapiclient.discovery import build

from rhuh_synthesis.drive_api import DriveRun

auth.authenticate_user()  # Colab asks once for permission to use the Drive API with this account
api = build("drive", "v3", cache_discovery=False)
source = DriveRun.open(api, (), SOURCE_RUN, LOCAL_SOURCE, create=False)  # run r2, by ID; never written
regions_run = DriveRun.open(api, (), REGIONS_RUN, LOCAL_REGIONS, create=False)  # never written
signals_run = DriveRun.open(api, (), SIGNALS_RUN, LOCAL_SIGNALS, create=False)  # never written
run_drive = DriveRun.open(api, (), OUT_RUN, LOCAL_RUN, create=True)
access = run_drive.preflight()  # a real write, update, read and removal on Drive before the long part
print("free GiB on Drive:", None if access["free_gib"] is None else round(access["free_gib"], 1))
STARTED = time.perf_counter()


def say(message):
    print(f"[{time.perf_counter() - STARTED:6.0f} s] {message}", flush=True)

## 2. Run r2's records and exports, the region record and the signal record, by ID

In [ ]:
for name in ("run.json", "split.json", "registration_qc.csv", "summary.json"):
    source.fetch(f"outputs/{name}")  # by ID, checked by MD5
outputs = Path(LOCAL_SOURCE) / "outputs"
r2 = json.loads((outputs / "summary.json").read_text(encoding="utf-8"))
assert r2["stopped_at"] is None and r2["exports"]["train"]["accepted"] == EXPECTED_PAIRS, "Run r2's summary differs from the card: stop and report"
say(f"run r2 training role: {r2['exports']['train']}")
files = source.fetch_folder("outputs/export/train")  # the training-role exports and their sidecars only
say(f"downloaded {files} files of the training-role exports")
regions_record = json.loads(regions_run.fetch("c4_lumiere_regions_v1.json").read_text(encoding="utf-8"))  # by ID
signals_record = json.loads(signals_run.fetch("outputs/c4_lumiere_signals.json").read_text(encoding="utf-8"))  # by ID
say(f"region record: {regions_record['summary']['rows']} pairs; signal record: {signals_record['summary']['rows']} pairs")

## 3. The five fills of every accepted training-role pair, and the write-once record

In [ ]:
from rhuh_synthesis.lumiere_scaffold_check import run_level_check

config = json.loads(Path("configs/experiments/EXP-2026-001.json").read_text(encoding="utf-8"))  # frozen, read only
summary, table, path, digest = run_level_check(
    outputs, config, regions_record, signals_record, Path(LOCAL_RUN) / f"{RUN_TAG}.json",
    source_run="EXP-2026-005 run r2", expected=EXPECTED_PAIRS, repo_root=WORK, progress=say,
)
report = run_drive.sync()
say(f"record {digest[:12]} written; Drive upload: {report}")

## 4. Confirm the upload and print the summary and the table

In [ ]:
assert run_drive.confirm(), f"Not every file is confirmed on Drive. The originals are in {LOCAL_RUN}: keep this session open and report"
print("Every file confirmed by the Drive server: MyDrive/" + "/".join(OUT_RUN))
print("SUMMARY")
print(json.dumps(summary, indent=1))
print("TABLE")
print(table)

## What to send back

Copy everything printed after `SUMMARY`, the `TABLE` included. They hold scores only, no patient name.

If a cell stops with an error, send the error message. The run stops on purpose when the files do not match their records: an error about the split, the export folder, a sidecar or a record, or a score that differs from the region record.